### Brute-force RAG on my WCAG knowledge base

## My accessibility auditor project. 

Redid the Insurellm employee exercise but on WCAG success criteria.

Goal today was NOT to build something impressive, it was to build the 
dumb version on purpose and actually feel where it breaks.

No embeddings, no vector DB. Just a dictionary and string matching.

#### The point was to earn the right to use embeddings later by first understanding what problem they solve.

In [ ]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [ ]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/a11y/success-criteria/*.md")

for filename in filenames:
    stem = Path(filename).stem            # e.g. "1.4.3 contrast-minimum"
    slug = stem.split(' ', 1)[-1]         # "contrast-minimum"
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()
    # register the file under each word of its slug, so "contrast" and "minimum" both match
    for word in slug.split('-'):
        knowledge.setdefault(word.lower(), content)

print(f"{len(filenames)} files loaded, {len(knowledge)} keyword keys")

In [ ]:
# The word 'contrast' legitimately belongs to THREE criteria.
# returns all 3.
contrast_files = knowledge['contrast']
print(f"'contrast' now maps to {len(contrast_files)} files:")
for c in contrast_files:
    print('', c.splitlines()[0])   # print the '# WCAG x.x.x — Title' header

In [ ]:
knowledge = {}
for filename in filenames:
    slug = Path(filename).stem.split(' ', 1)[-1]
    content = open(filename, encoding='utf-8').read()
    for word in slug.split('-'):
        knowledge.setdefault(word.lower(), []).append(content)  # collect ALL, not first

In [ ]:
knowledge.keys()

In [ ]:
knowledge

In [ ]:
# Strips punctuation, lowercases, splits the question into words, 
# and returns any file whose keyword appears in the question. 
# That's the whole "retriever" it's a lookup table, like a book index.

def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    seen, hits = set(), []
    for word in words:
        for content in knowledge.get(word, []):   # each key -> list of files
            if content not in seen:                # dedupe by file content
                seen.add(content)
                hits.append(content)
    return hits

### The bruteforce retriever- pure keyword matching

In [ ]:
# 'contrast' now pulls all three contrast criteria
ctx = get_relevant_context("What are the rules about contrast?")
print(f"{len(ctx)} file(s) matched:")
for c in ctx:
    print('   ', c.splitlines()[0])
    print(c[0][:300] if c else "no match")

Same intent with different words

In [ ]:
# Same meaning as 1.4.3, but no shared keyword -> still zero hits
print("paraphrased:", len(get_relevant_context("my text is hard to read against the background")))

# Only works when the user says the magic word
print("exact keyword:", len(get_relevant_context("contrast rules")))

In [ ]:
SYSTEM_PREFIX = """
You are a WCAG 2.2 assistant. Answer ONLY using the context provided below.
If the context does not contain the answer, reply exactly:
"That criterion isn't in my retrieved context."
Do not use any outside knowledge. Do not mention criteria that are not in the context.
Always cite the SC number from the context.

Context:
"""

def additional_context(message):
    ctx = get_relevant_context(message)
    if not ctx:
        return "No relevant context found."
    return "\n\n".join(ctx)

print(additional_context("contrast")[:400])

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    print(SYSTEM_PREFIX + additional_context("contrast"))
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
msg = "contrast"
full = SYSTEM_PREFIX + additional_context(msg)
print("=== LENGTH:", len(full), "===")
print(repr(full[-600:]))   # repr shows \n literally so we can see escaping

In [ ]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

In [ ]:
print(type(retriever) if 'retriever' in dir() else "no 'retriever' variable")
import inspect
print(inspect.getsource(chat))

In [ ]:
# 1. Show retrieval returned nothing for that query
print("matched files:", len(get_relevant_context("my text is hard to read against the background")))

# 2. Show the model answers even with ZERO context — pure memory
print(chat("my text is hard to read against the background", []))